# Lesson 03 — SQLAlchemy ORM Models

**Run this in Google Colab** — every student gets the same environment.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mpalomera/learning-sql/blob/master/lessons/06-sqlalchemy-orm-migrations/03_sqlalchemy_models.ipynb)

---

## Step 1: Install dependencies

In [2]:
!pip install sqlalchemy oracledb -q

You should consider upgrading via the '/Users/baltazarservin/Documents/School/TEC/6to_Semestre/Software_Development/Advanced_Database/advanced_database_notes/.venv/bin/python3 -m pip install --upgrade pip' command.


## Step 2: Define ORM models

In [3]:
from datetime import datetime
from sqlalchemy import (
    create_engine, Column, Integer, String,
    Text, ForeignKey, DateTime, func
)
from sqlalchemy.orm import declarative_base, relationship, Session

Base = declarative_base()

class Team(Base):
    __tablename__ = "teams"
    id          = Column(Integer, primary_key=True)
    name        = Column(String(50), nullable=False, unique=True)
    description = Column(String(200))
    created_at  = Column(DateTime, server_default=func.current_timestamp())
    users = relationship("User", back_populates="team")

    def __repr__(self):
        return f"<Team(id={self.id}, name='{self.name}')>"

class User(Base):
    __tablename__ = "users"
    id         = Column(Integer, primary_key=True)
    username   = Column(String(50), nullable=False, unique=True)
    email      = Column(String(100), nullable=False)
    full_name  = Column(String(100))
    team_id    = Column(Integer, ForeignKey("teams.id"))
    created_at = Column(DateTime, server_default=func.current_timestamp())
    team     = relationship("Team", back_populates="users")
    tasks    = relationship("Task", back_populates="assignee")
    comments = relationship("Comment", back_populates="user")

    def __repr__(self):
        return f"<User(id={self.id}, username='{self.username}')>"

class Task(Base):
    __tablename__ = "tasks"
    id           = Column(Integer, primary_key=True)
    title        = Column(String(200), nullable=False)
    description  = Column(String(1000))
    status       = Column(String(20), default="open")
    assigned_to  = Column(Integer, ForeignKey("users.id"))
    created_at   = Column(DateTime, server_default=func.current_timestamp())
    updated_at   = Column(DateTime, onupdate=func.current_timestamp())
    assignee = relationship("User", back_populates="tasks")
    comments = relationship("Comment", back_populates="task", cascade="all, delete-orphan")

    def __repr__(self):
        return f"<Task(id={self.id}, title='{self.title}', status='{self.status}')>"

# Exercise 1 — Comment model
class Comment(Base):
    __tablename__ = "comments"

    id         = Column(Integer, primary_key=True, autoincrement=True)
    task_id    = Column(Integer, ForeignKey("tasks.id", ondelete="CASCADE"), nullable=False)
    user_id    = Column(Integer, ForeignKey("users.id"), nullable=False)
    content    = Column(Text, nullable=False)
    created_at = Column(DateTime, default=datetime.utcnow)

    task = relationship("Task", back_populates="comments")
    user = relationship("User", back_populates="comments")

    def __repr__(self):
        return f"<Comment(id={self.id}, task_id={self.task_id}, user_id={self.user_id})>"

print("✅ Models defined: Team, User, Task, Comment")

✅ Models defined: Team, User, Task, Comment


## Step 3: Connect and query with ORM

In [ ]:
# ============================================================
# CONFIGURATION — replace with your FreeSQL credentials
# ============================================================
USERNAME = "heehee"
PASSWORD = "heehee"   # <-- REPLACE THIS
DSN = "heehee"

engine = create_engine(
    "oracle+oracledb://:@",
    connect_args={
        "user": USERNAME,
        "password": PASSWORD,
        "dsn": DSN
    }
)

with Session(engine) as session:
    print("🏢 Teams:")
    for team in session.query(Team).all():
        print(f"   {team}")
        for user in team.users:
            print(f"      -> {user.full_name} ({user.username})")

    print("\n📝 Tasks with assignees:")
    for task in session.query(Task).all():
        assignee = task.assignee.full_name if task.assignee else "Unassigned"
        print(f"   {task.title} → {assignee}")

print("\n✅ ORM models working! Relationships navigate automatically.")

🏢 Teams:
   <Team(id=1, name='Engineering')>
      -> Alice Smith (alice_dev)
      -> Bob Jones (bob_dev)
   <Team(id=2, name='Product')>
      -> Carol White (carol_pm)

📝 Tasks with assignees:
   Fix login bug → Alice Smith
   Design new dashboard → Carol White
   Update dependencies → Bob Jones

✅ ORM models working! Relationships navigate automatically.


# Lesson 03 — Alembic Migrations (Pure Python)


In [8]:
!pip install sqlalchemy oracledb alembic -q

You should consider upgrading via the '/Users/baltazarservin/Documents/School/TEC/6to_Semestre/Software_Development/Advanced_Database/advanced_database_notes/.venv/bin/python3 -m pip install --upgrade pip' command.


In [9]:
import os
from alembic.config import Config
from alembic import command

# Create a minimal alembic.ini in memory
alembic_cfg = Config()
alembic_cfg.set_main_option('script_location', '/content/alembic')
alembic_cfg.set_main_option('sqlalchemy.url', 'oracle+oracledb://:@')

# Initialize the migration directory
!mkdir -p /content/alembic/versions

# Write a minimal env.py
env_py = '''
from logging.config import fileConfig
from sqlalchemy import engine_from_config
from alembic import context

# This is the Alembic Config object
config = context.config

# Add your model's MetaData object here for 'autogenerate' support
from __main__ import Base
target_metadata = Base.metadata

def run_migrations_offline():
    url = config.get_main_option('sqlalchemy.url')
    context.configure(url=url, target_metadata=target_metadata, literal_binds=True)
    with context.begin_transaction():
        context.run_migrations()

def run_migrations_online():
    connectable = engine_from_config(
        config.get_section(config.config_ini_section),
        prefix='sqlalchemy.',
        connect_args={'user': "''' + USERNAME + '''", 'password': "''' + PASSWORD + '''", 'dsn': "''' + DSN + '''"}
    )
    with connectable.connect() as connection:
        context.configure(connection=connection, target_metadata=target_metadata)
        with context.begin_transaction():
            context.run_migrations()

if context.is_offline_mode():
    run_migrations_offline()
else:
    run_migrations_online()
'''

with open('/content/alembic/env.py', 'w') as f:
    f.write(env_py)

# Write a minimal script.py.mako
script_template = '''"""${message}

Revision ID: ${up_revision}
Revises: ${down_revision | comma,n}
Create Date: ${create_date}
"""

from alembic import op
import sqlalchemy as sa
${imports if imports else ""}

# revision identifiers, used by Alembic.
revision = ${repr(up_revision)}
down_revision = ${repr(down_revision)}
branch_labels = ${repr(branch_labels)}
depends_on = ${repr(depends_on)}

def upgrade():
${upgrades if upgrades else "    pass"}

def downgrade():
${downgrades if downgrades else "    pass"}
'''

with open('/content/alembic/script.py.mako', 'w') as f:
    f.write(script_template)

print('✅ Alembic initialized in /content/alembic/')

mkdir: /content: Read-only file system


FileNotFoundError: [Errno 2] No such file or directory: '/content/alembic/env.py'

In [10]:
# Generate migration from models vs current database state
command.revision(alembic_cfg, autogenerate=True, message='Initial schema')

# Show what was generated
import glob
migration_files = sorted(glob.glob('/content/alembic/versions/*.py'))
print('Generated migrations:')
for f in migration_files:
    print(f'  {f}')

CommandError: Path doesn't exist: /content/alembic.  Please use the 'init' command to create a new scripts folder.

In [64]:
# Read and display the latest migration
latest = migration_files[1]
with open(latest) as f:
    content = f.read()

print(content)


"""Initial schema

Revision ID: dc62f0e11bfa
Revises: d67fc4608479
Create Date: 2026-05-12 13:49:07.882812
"""

from alembic import op
import sqlalchemy as sa


# revision identifiers, used by Alembic.
revision = 'dc62f0e11bfa'
down_revision = 'd67fc4608479'
branch_labels = None
depends_on = None

def upgrade():
# ### commands auto generated by Alembic - please adjust! ###
    op.add_column('teams', sa.Column('priority', sa.String(length=20), nullable=True))
    op.add_column('teams', sa.Column('due_date', sa.DateTime(), nullable=True))
    op.add_column('teams', sa.Column('tags', sa.String(length=500), nullable=True))
    # ### end Alembic commands ###

def downgrade():
# ### commands auto generated by Alembic - please adjust! ###
    op.drop_column('teams', 'tags')
    op.drop_column('teams', 'due_date')
    op.drop_column('teams', 'priority')
    # ### end Alembic commands ###



In [ ]:
team = Team()

In [65]:

command.upgrade(alembic_cfg, 'head')
print('✅ Migration applied! Tables created.')

✅ Migration applied! Tables created.


In [66]:
# Rollback one migration
command.downgrade(alembic_cfg, '-1')
print('✅ Downgraded by 1. New columns removed.')

✅ Downgraded by 1. New columns removed.


In [ ]:
import glob
import os

# Delete all generated migration files
migration_files = glob.glob('/content/alembic/versions/*.py')

for f in migration_files:
    os.remove(f)
    print(f"Deleted: {f}")

# Exercise 2 — Migration Creation

Generate an Alembic migration for the `Comment` model, inspect it, and apply it.

In [ ]:
# Exercise 2 — Step 1: Generate a migration for the Comment model
command.revision(
    alembic_cfg,
    autogenerate=True,
    message="add comments table"
)
print("✅ Migration generated.")

In [ ]:
# Exercise 2 — Step 2: Inspect the generated migration
import glob

migration_files = sorted(
    glob.glob('/content/alembic/versions/*.py')
)

print("Migration files:")
for f in migration_files:
    print(f"  {f}")

# Open and print the latest migration
latest = migration_files[-1]
print(f"\n--- Contents of {latest} ---\n")
with open(latest) as f:
    print(f.read())

In [ ]:
# Exercise 2 — Step 3: Apply the migration
command.upgrade(alembic_cfg, 'head')
print("✅ Migration applied! comments table created.")

# Questions:
# 1. What does upgrade() do?
#    upgrade() runs the forward migration — in this case, it creates the comments table
#    with all its columns, foreign keys, and constraints.
#
# 2. What does downgrade() do?
#    downgrade() reverses the migration — it drops the comments table entirely,
#    undoing everything upgrade() did.
#
# 3. What happens if you downgrade this migration?
#    The comments table is dropped from the database, and all comment rows are
#    permanently deleted along with it.

# Exercise 3 — CRUD Challenge

Script that creates a DevOps team, diana_ops user, 3 prioritized tasks, prints the count, closes the top task, and deletes the lowest.

In [ ]:
with Session(engine) as session:
    # 1. Create DevOps team
    devops_team = Team(name="DevOps", description="Infrastructure and operations team")
    session.add(devops_team)
    session.flush()
    print(f"Created: {devops_team}")

    # 2. Create diana_ops user linked via relationship
    diana = User(
        username="diana_ops",
        email="diana@example.com",
        full_name="Diana Ops",
        team_id=devops_team.id
    )
    session.add(diana)
    session.flush()
    print(f"Created: {diana}")

    # 3. Create 3 tasks with different priorities (encoded in title)
    task_high = Task(
        title="[HIGH] Deploy Kubernetes cluster",
        description="Set up production k8s cluster",
        status="open",
        assigned_to=diana.id
    )
    task_med = Task(
        title="[MEDIUM] Configure CI/CD pipeline",
        description="Set up GitHub Actions workflows",
        status="open",
        assigned_to=diana.id
    )
    task_low = Task(
        title="[LOW] Update server documentation",
        description="Write runbook for server maintenance",
        status="open",
        assigned_to=diana.id
    )
    session.add_all([task_high, task_med, task_low])
    session.flush()

    # 4. Print task count using relationship
    print(f"\nTask count for {diana.username}: {len(diana.tasks)}")
    for t in diana.tasks:
        print(f"   - {t.title} [{t.status}]")

    # 5. Close the high-priority task
    task_high.status = "closed"
    print(f"\nClosed: {task_high.title}")

    # 6. Delete the lowest-priority task
    session.delete(task_low)
    print(f"Deleted: {task_low.title}")

    session.commit()
    print("\n✅ Exercise 3 complete!")

# Exercise 4 — Migration Rollback

You added a bad column (`estimated_hours`) and already applied the migration.
Roll it back programmatically.

In [ ]:
import glob, os

# Create a migration for the bad column manually (autogenerate would need model change)
command.revision(alembic_cfg, message="add estimated_hours to tasks")

migration_files = sorted(glob.glob('/content/alembic/versions/*.py'))
latest = migration_files[-1]
print(f"Created migration: {latest}")

# Patch the migration to add/drop the column
with open(latest) as f:
    content = f.read()

content = content.replace(
    "def upgrade():\n    pass",
    "def upgrade():\n    op.add_column('tasks', sa.Column('estimated_hours', sa.Integer(), nullable=True))"
)
content = content.replace(
    "def downgrade():\n    pass",
    "def downgrade():\n    op.drop_column('tasks', 'estimated_hours')"
)

with open(latest, 'w') as f:
    f.write(content)

# Apply it
command.upgrade(alembic_cfg, 'head')
print("✅ Migration applied — estimated_hours column added to tasks.")

In [ ]:
# Exercise 4 — Rollback the bad migration
command.downgrade(alembic_cfg, "-1")
print("✅ Downgraded by 1.")
print("   What happened to the column? → It was dropped from the tasks table.")
print("   What happened to the data?   → Any values stored in estimated_hours are permanently lost.")

# Exercise 5 — Concept Check

Answer the following questions about ORM and migrations.

In [ ]:
concept_check = """
Exercise 5 — Concept Check Answers
====================================

1. Why use ORM instead of raw SQL?
   ORM maps tables to Python classes, so you work with objects instead of strings.
   You get type safety, automatic SQL-injection escaping, relationship navigation
   (task.assignee, team.users), and portability across database engines — all without
   writing SQL by hand.

2. Why use migrations?
   Migrations version-control every schema change as code. Every developer and every
   environment (dev/staging/prod) applies the same changes in the same order, making
   the schema history auditable and reversible.

3. When would you rollback?
   When a migration introduces a bug or breaking change — for example, a bad constraint,
   a column that causes query failures, or a schema change that conflicts with the
   running application code.

4. Difference between add() and commit()?
   session.add() stages an object in the session's in-memory unit of work.
   session.commit() flushes all pending changes and writes them permanently to the
   database. Without commit(), nothing is actually saved to disk.

5. Why are relationships useful?
   Relationships let you navigate between related objects (task.assignee, team.users)
   without writing JOIN queries manually. SQLAlchemy loads related objects as needed,
   keeping application code clean and readable.
"""
print(concept_check)